# DINOv2-T5 unpruned base: decoder-start visual cross-attention

Enable Internet and one GPU or T4 x2 for measurement. All artifacts download from pinned
Hugging Face revisions; raw images and the training repository are not required.
**Cell 1 clones/pulls the entire Visualize repository.** Push code to the configured branch first.
Source updates preserve the separate output directory and are blocked while workers run.

Measure all 16,817 test questions, batch **32/GPU**, FP32. CUDA OOM halves the batch.
Query is decoder-start `<pad>` (ID 0) with mask 1, predicting the first answer token.
Mean eight heads, then equal-weight questions. Softmax spans all valid encoder keys;
the 384 image positions are not renormalized. Four decoder layers are measured.
These encoder positions already contain bidirectionally mixed information, not raw patches.

**New/resumed run:** run all cells with `RUN_MODE='auto'`.
**Redraw only:** set `RUN_MODE='plot'`, run cells 1-2, then 6-10 (no GPU/model/cache scan).
Partial resume requires the same GPU count and compatible measurement code/library versions.
Updating only plotting does not invalidate saved measurements.
tqdm shows validation and worker progress; interrupting the wait cell leaves workers running.
Figures use shared log attention scales; histogram attention bins are linear and count color is log.
The last cell downloads a ZIP containing figures, metadata and the measurement caption only.


In [ ]:
# 1. Pull the complete Visualize source before importing inference code.
# Push updates first; finish all attention workers before rerunning this cell.
import os
from pathlib import Path
SOURCE_REPOSITORY = 'https://github.com/huyvanzzz/Log_Metrics.git'
SOURCE_BRANCH = 'smolvlm-attention'
SOURCE = Path('/kaggle/working/visualize-github-source')
OUTPUT = Path('/kaggle/working/dinov2_attention_outputs')
RUN_MODE = 'auto'  # auto, measure, resume, or plot
BATCH_PER_GPU = 32  # Fixed initial batch; CUDA OOM halves it.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'

"""Standard-library-only Git setup, embedded verbatim in both Kaggle notebooks."""
import hashlib
import importlib
import importlib.util
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile


def ensure_no_workers(output_roots):
    for name in ('smolvlm_attention.runtime', 'smolvlm_attention.generated_runtime',
                 'dinov2_attention.runtime', 'smolvlm_probing.runtime',
                 'smolvlm_attention.sample_runtime', 'smolvlm_topk_benchmark.runtime'):
        module = sys.modules.get(name)
        if module is not None and any(process.poll() is None for process in getattr(module, '_PROCESSES', {}).values()):
            raise RuntimeError('A measurement worker is still running. Wait for it to finish before pulling source.')
    # The canonical probing output is shared by all notebooks. Also inspect it
    # after a kernel restart, when the new runtime is no longer in sys.modules.
    roots = set(map(Path, output_roots)) | {Path('/kaggle/working/smolvlm_probing_outputs'),
                                          Path('/kaggle/working/smolvlm_prompt_attention_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64_two_waves')}
    for root in roots:
        for path in (Path(root) / 'logs').glob('*.pid.json'):
            pid = json.loads(path.read_text(encoding='utf-8'))['pid']
            try:
                os.kill(pid, 0)
            except (OSError, ProcessLookupError):
                continue
            raise RuntimeError(f'A measurement worker PID {pid} may still be running ({path}). Finish it before pulling.')


def sync_source(repository, branch, directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    def git(*args):
        return subprocess.check_output(['git', '-C', str(directory), *args], text=True).strip()
    if directory.exists():
        if not (directory / '.git').exists():
            raise RuntimeError(f'{directory} is not a Git checkout. Choose a new SOURCE directory.')
        if git('remote', 'get-url', 'origin').rstrip('/') != str(repository).rstrip('/'):
            raise RuntimeError('Existing source checkout has a different origin; choose a new SOURCE directory.')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Source checkout has local changes. Preserve them before pulling; no files were reset.')
        if git('branch', '--show-current') != branch:
            raise RuntimeError('Checkout branch differs from SOURCE_BRANCH. Use the matching branch or a new SOURCE directory.')
        subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', 'origin', branch], check=True)
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
                        str(repository), str(directory)], check=True)
    # Keep the historical source_archive_sha256 field reproducible without embedding stale source.
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for package in packages:
            folder = directory / package
            if not folder.is_dir():
                raise RuntimeError(f'Missing source package after pull: {package}')
            for path in sorted(folder.iterdir()):
                if path.suffix in ('.py', '.txt'):
                    entry = zipfile.ZipInfo(path.relative_to(directory).as_posix(), (2026, 1, 1, 0, 0, 0))
                    entry.compress_type = zipfile.ZIP_DEFLATED
                    archive.writestr(entry, path.read_bytes())
    return dict(commit=git('rev-parse', 'HEAD'), sha256=hashlib.sha256(buffer.getvalue()).hexdigest())


def activate_source(directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    for package in packages:
        for path in (directory / package).glob('*.py'):
            Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + '.')
               for package in packages):
            del sys.modules[name]
    sys.path[:] = [str(directory)] + [entry for entry in sys.path if entry != str(directory)]
    importlib.invalidate_caches()

ensure_no_workers([Path('/kaggle/working/attention_outputs'),
                   Path('/kaggle/working/generated_attention_outputs'), OUTPUT])
identity = sync_source(SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE,
                       packages=('dinov2_attention', 'smolvlm_attention'))
SOURCE_COMMIT, SOURCE_SHA256 = identity['commit'], identity['sha256']
activate_source(SOURCE, packages=('dinov2_attention', 'smolvlm_attention'))
from dinov2_attention.runtime import resolve_run_mode
BOOT_MODE = resolve_run_mode(OUTPUT, RUN_MODE)
if BOOT_MODE != 'plot':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                    '-r', str(SOURCE/'dinov2_attention/requirements.txt')], check=True)
    import torch, transformers, tokenizers
    if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
        raise RuntimeError('Restart the kernel after installing pinned packages, then rerun setup.')
from dinov2_attention.config import BATCH_SIZE
if BATCH_PER_GPU != BATCH_SIZE:
    raise ValueError('This experiment uses batch 32/GPU; only CUDA OOM reduces it.')
RUN_CONFIG = OUTPUT/'run.json'
print(dict(source=str(SOURCE), commit=SOURCE_COMMIT, output=str(OUTPUT),
           mode=BOOT_MODE, batch_per_gpu=BATCH_SIZE,
           gpus=('not required for plot' if BOOT_MODE == 'plot' else
                 [torch.cuda.get_device_name(i) for i in range(min(2,torch.cuda.device_count()))])))


In [ ]:
# 2. Decide whether to measure, resume, or only redraw saved results.
from dinov2_attention.runtime import resolve_run_mode
MODE = resolve_run_mode(OUTPUT, RUN_MODE)
RUN_CONFIG = OUTPUT/'run.json'
print('Selected mode:', MODE)


In [ ]:
# 3. Download/validate once and prepare all test questions. Plot mode skips this.
if MODE != 'plot':
    from dinov2_attention.runtime import prepare_run
    RUN_CONFIG = prepare_run(OUTPUT, mode=MODE, source_commit=SOURCE_COMMIT,
                             source_sha256=SOURCE_SHA256)
print('Run configuration:', RUN_CONFIG)


In [ ]:
# 4. Launch one/two independent workers; each strict-loads and preflights the model.
if MODE != 'plot':
    from dinov2_attention.runtime import launch_workers
    WORKERS = launch_workers(RUN_CONFIG)


In [ ]:
# 5. Live tqdm per GPU and total. Rerun this cell to reconnect after interruption.
if MODE != 'plot':
    from dinov2_attention.runtime import wait_workers
    wait_workers(RUN_CONFIG)


In [ ]:
# 6. Verify full coverage and load saved attention without loading the model.
from dinov2_attention.runtime import load_results
from dinov2_attention.plotting import shared_scales
from dinov2_attention.config import CAMERA_ORDER
MEAN, METADATA = load_results(RUN_CONFIG, plot_source_commit=SOURCE_COMMIT,
                              plot_source_sha256=SOURCE_SHA256)
SCALES = shared_scales(MEAN)
FIGURES = OUTPUT/'figures'
TITLE = 'DINOv2-T5 base: decoder-start visual cross-attention'
print('Measured questions:', METADATA['count'], 'Mean shape:', MEAN.shape)


In [ ]:
# 7. Four layers, 2x2 panels; token attention Y and color use log scales.
from IPython.display import display, Image, FileLink
from dinov2_attention.plotting import plot_token_atlas
paths = plot_token_atlas(MEAN, CAMERA_ORDER, FIGURES, title=TITLE, scales=SCALES)
display(Image(filename=str(paths[0])))


In [ ]:
# 8. Four layers, 2x2 panels; six unflipped 8x8 camera maps per horizontal row.
from dinov2_attention.plotting import plot_spatial_atlas
paths = plot_spatial_atlas(MEAN, CAMERA_ORDER, FIGURES, title=TITLE, scales=SCALES)
display(Image(filename=str(paths[0])))


In [ ]:
# 9. Histogram: 384 averaged visual positions/layer, logarithmic count color.
from dinov2_attention.plotting import plot_layer_histogram
paths = plot_layer_histogram(MEAN, CAMERA_ORDER, FIGURES, title=TITLE, scales=SCALES)
display(Image(filename=str(paths[0])))


In [ ]:
# 10. Download six PNG/PDF figures, metadata, and the measurement caption.
from dinov2_attention.plotting import make_figures_zip
ZIP = make_figures_zip(FIGURES, OUTPUT/'dinov2_base_attention_figures.zip', METADATA)
display(FileLink(str(ZIP)))
